In [1]:
%%capture
%cd /kaggle/input/omniparser/pytorch/default/1/OmniParser
!pip install -r requirements.txt
!pip install openai ultralytics firebase-admin -q

In [10]:
import firebase_admin
from firebase_admin import credentials, db
from kaggle_secrets import UserSecretsClient
import json

secrets = UserSecretsClient()
cred_json  = json.loads(secrets.get_secret("FIREBASE_CREDENTIALS"))
db_url     = secrets.get_secret("FIREBASE_DATABASE_URL")

cred = credentials.Certificate(cred_json)
firebase_admin.initialize_app(cred, {'databaseURL': db_url})
print("Firebase connected.")

Firebase connected.


In [11]:
from utils import get_som_labeled_img, check_ocr_box, get_caption_model_processor, get_yolo_model
import torch
from ultralytics import YOLO
from PIL import Image
device = 'cuda'

som_model = get_yolo_model(model_path='/kaggle/input/omniparser/pytorch/default/1/OmniParser/weights/icon_detect/best.pt')
som_model.to(device)
print('model to {}'.format(device))

model to cuda


In [13]:
# two choices for caption model: fine-tuned blip2 or florence2

# caption_model_processor = get_caption_model_processor(model_name="blip2", model_name_or_path="weights/icon_caption_blip2", device=device)
caption_model_processor = get_caption_model_processor(model_name="florence2", model_name_or_path="weights/icon_caption_florence", device=device)

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [14]:
som_model.device, type(som_model)

(device(type='cuda', index=0), ultralytics.models.yolo.model.YOLO)

## Core Utilities — BoxAnnotator

In [15]:
from typing import List, Optional, Union, Tuple
import cv2
import numpy as np
from supervision.detection.core import Detections
from supervision.draw.color import Color, ColorPalette


def box_area(box):
    return (box[2] - box[0]) * (box[3] - box[1])

def intersection_area(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    return max(0, x2 - x1) * max(0, y2 - y1)

def IoU(box1, box2, return_max=True):
    inter = intersection_area(box1, box2)
    union = box_area(box1) + box_area(box2) - inter
    r1 = inter / box_area(box1) if box_area(box1) > 0 else 0
    r2 = inter / box_area(box2) if box_area(box2) > 0 else 0
    return max(inter / union, r1, r2) if return_max else inter / union

def get_optimal_label_pos(text_padding, text_width, text_height,
                           x1, y1, x2, y2, detections, image_size):
    def get_is_overlap(detections, bx1, by1, bx2, by2, image_size):
        if bx1 < 0 or bx2 > image_size[0] or by1 < 0 or by2 > image_size[1]:
            return True
        for i in range(len(detections)):
            det = detections.xyxy[i].astype(int)
            if IoU([bx1, by1, bx2, by2], det) > 0.3:
                return True
        return False

    candidates = [
        (x1 + text_padding, y1 - text_padding,
         x1, y1 - 2*text_padding - text_height,
         x1 + 2*text_padding + text_width, y1),
        (x1 - text_padding - text_width, y1 + text_padding + text_height,
         x1 - 2*text_padding - text_width, y1,
         x1, y1 + 2*text_padding + text_height),
        (x2 + text_padding, y1 + text_padding + text_height,
         x2, y1,
         x2 + 2*text_padding + text_width, y1 + 2*text_padding + text_height),
        (x2 - text_padding - text_width, y1 - text_padding,
         x2 - 2*text_padding - text_width, y1 - 2*text_padding - text_height,
         x2, y1),
    ]
    for tx, ty, tbx1, tby1, tbx2, tby2 in candidates:
        if not get_is_overlap(detections, tbx1, tby1, tbx2, tby2, image_size):
            return tx, ty, tbx1, tby1, tbx2, tby2
    return candidates[-1]


class BoxAnnotator:
    def __init__(self, color=ColorPalette.DEFAULT, thickness=3,
                 text_color=Color.BLACK, text_scale=0.5,
                 text_thickness=2, text_padding=10, avoid_overlap=True):
        self.color = color
        self.thickness = thickness
        self.text_color = text_color
        self.text_scale = text_scale
        self.text_thickness = text_thickness
        self.text_padding = text_padding
        self.avoid_overlap = avoid_overlap

    def annotate(self, scene, detections, labels=None,
                 skip_label=False, image_size=None):
        font = cv2.FONT_HERSHEY_SIMPLEX
        for i in range(len(detections)):
            x1, y1, x2, y2 = detections.xyxy[i].astype(int)
            class_id = detections.class_id[i] if detections.class_id is not None else None
            idx = class_id if class_id is not None else i
            color = (self.color.by_idx(idx)
                     if isinstance(self.color, ColorPalette) else self.color)
            cv2.rectangle(scene, (x1, y1), (x2, y2), color.as_bgr(), self.thickness)
            if skip_label:
                continue
            text = (f"{class_id}"
                    if (labels is None or len(detections) != len(labels))
                    else labels[i])
            tw, th = cv2.getTextSize(text, font, self.text_scale, self.text_thickness)[0]
            if not self.avoid_overlap:
                tx, ty = x1 + self.text_padding, y1 - self.text_padding
                tbx1, tby1 = x1, y1 - 2*self.text_padding - th
                tbx2, tby2 = x1 + 2*self.text_padding + tw, y1
            else:
                tx, ty, tbx1, tby1, tbx2, tby2 = get_optimal_label_pos(
                    self.text_padding, tw, th, x1, y1, x2, y2,
                    detections, image_size)
            cv2.rectangle(scene, (tbx1, tby1), (tbx2, tby2),
                          color.as_bgr(), cv2.FILLED)
            r, g, b = color.as_rgb()
            lum = 0.299*r + 0.587*g + 0.114*b
            text_col = (0, 0, 0) if lum > 160 else (255, 255, 255)
            cv2.putText(scene, text, (tx, ty), font, self.text_scale,
                        text_col, self.text_thickness, cv2.LINE_AA)
        return scene

print('BoxAnnotator defined.')

BoxAnnotator defined.


## Parser Functions

In [16]:
import io
import base64
import time
import torch
import numpy as np
from PIL import Image
from torchvision.ops import box_convert
from torchvision.transforms import ToPILImage
import supervision as sv
import easyocr

# ── OCR readers (initialised once) ──────────────────────────────────────────
print('Loading EasyOCR...')
_easyocr_reader = easyocr.Reader(['en'])

try:
    from paddleocr import PaddleOCR
    print('Loading PaddleOCR...')
    _paddle_ocr = PaddleOCR(
        lang='en', use_angle_cls=False, use_gpu=False,
        show_log=False, max_batch_size=1024,
        use_dilation=True, det_db_score_mode='slow',
        rec_batch_num=1024)
    PADDLE_AVAILABLE = True
    print('PaddleOCR ready.')
except Exception as e:
    print(f'PaddleOCR unavailable ({e}); falling back to EasyOCR only.')
    PADDLE_AVAILABLE = False


# ── Helper coordinate extractor ──────────────────────────────────────────────
def _get_xyxy(item):
    return int(item[0][0]), int(item[0][1]), int(item[2][0]), int(item[2][1])


# ── OCR ─────────────────────────────────────────────────────────────────────
def check_ocr_box(image_path, output_bb_format='xyxy',
                  easyocr_args=None, use_paddleocr=False):
    if use_paddleocr and PADDLE_AVAILABLE:
        thresh = (easyocr_args or {}).get('text_threshold', 0.5)
        result = _paddle_ocr.ocr(image_path, cls=False)[0]
        coord = [item[0] for item in result if item[1][1] > thresh]
        text  = [item[1][0] for item in result if item[1][1] > thresh]
    else:
        args   = easyocr_args or {}
        result = _easyocr_reader.readtext(image_path, **args)
        coord  = [item[0] for item in result]
        text   = [item[1] for item in result]
    if output_bb_format == 'xyxy':
        bb = [_get_xyxy(c) for c in coord]
    else:
        bb = [(int(c[0][0]), int(c[0][1]),
               int(c[2][0]-c[0][0]), int(c[2][1]-c[0][1])) for c in coord]
    return (text, bb), None


# ── YOLO prediction ─────────────────────────────────────────────────────────
def predict_yolo(model, image_path, box_threshold, imgsz,
                 scale_img, iou_threshold=0.7):
    kw = dict(source=image_path, conf=box_threshold, iou=iou_threshold)
    if scale_img:
        kw['imgsz'] = imgsz
    result  = model.predict(**kw)
    boxes   = result[0].boxes.xyxy
    conf    = result[0].boxes.conf
    phrases = [str(i) for i in range(len(boxes))]
    return boxes, conf, phrases


# ── Overlap removal ──────────────────────────────────────────────────────────
def _iou_elem(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter   = max(0, x2-x1) * max(0, y2-y1)
    a1, a2  = box_area(b1), box_area(b2)
    union   = a1 + a2 - inter + 1e-6
    r1 = inter/a1 if a1 > 0 else 0
    r2 = inter/a2 if a2 > 0 else 0
    return max(inter/union, r1, r2)

def _is_inside(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter   = max(0, x2-x1) * max(0, y2-y1)
    a1      = box_area(b1)
    return (inter / a1) > 0.95 if a1 > 0 else False

def remove_overlap_new(boxes, iou_threshold, ocr_bbox=None):
    filtered = list(ocr_bbox) if ocr_bbox else []
    for i, e1 in enumerate(boxes):
        b1 = e1['bbox']
        valid = all(
            not (_iou_elem(b1, e2['bbox']) > iou_threshold
                 and box_area(b1) > box_area(e2['bbox']))
            for j, e2 in enumerate(boxes) if i != j
        )
        if not valid:
            continue
        added = False
        if ocr_bbox:
            for e3 in list(ocr_bbox):
                b3 = e3['bbox']
                if _is_inside(b3, b1):        # OCR inside icon → promote
                    try:
                        filtered.append({'type': 'text', 'bbox': b1,
                                         'interactivity': True,
                                         'content': e3['content']})
                        filtered.remove(e3)
                    except Exception:
                        pass
                    added = True; break
                elif _is_inside(b1, b3):      # icon inside OCR → keep both
                    try:
                        filtered.append({'type': 'icon', 'bbox': b1,
                                         'interactivity': True, 'content': None})
                        filtered.remove(e3)
                    except Exception:
                        pass
                    added = True; break
        if not added:
            filtered.append({'type': 'icon', 'bbox': b1,
                             'interactivity': True, 'content': None})
    return filtered


# ── Caption model loader ─────────────────────────────────────────────────────
def get_caption_model_processor(model_name, model_name_or_path, device=None):
    if not device:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.float16 if device != 'cpu' else torch.float32
    if model_name == 'blip2':
        from transformers import Blip2Processor, Blip2ForConditionalGeneration
        processor = Blip2Processor.from_pretrained('Salesforce/blip2-opt-2.7b')
        model = Blip2ForConditionalGeneration.from_pretrained(
            model_name_or_path, device_map=None, torch_dtype=dtype).to(device)
    elif model_name == 'florence2':
        from transformers import AutoProcessor, AutoModelForCausalLM
        processor = AutoProcessor.from_pretrained(
            'microsoft/Florence-2-base', trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path, torch_dtype=dtype,
            trust_remote_code=True).to(device)
    return {'model': model, 'processor': processor}

def get_yolo_model(model_path):
    from ultralytics import YOLO
    return YOLO(model_path)


# ── Icon captioning ──────────────────────────────────────────────────────────
@torch.inference_mode()
def get_parsed_content_icon(filtered_boxes, starting_idx, image_source,
                             caption_model_processor, prompt=None, batch_size=32):
    to_pil = ToPILImage()
    non_ocr = filtered_boxes[starting_idx:] if starting_idx >= 0 else filtered_boxes
    crops = []
    for coord in non_ocr:
        xmin = int(coord[0] * image_source.shape[1])
        xmax = int(coord[2] * image_source.shape[1])
        ymin = int(coord[1] * image_source.shape[0])
        ymax = int(coord[3] * image_source.shape[0])
        crops.append(to_pil(image_source[ymin:ymax, xmin:xmax, :]))

    model, processor = caption_model_processor['model'], caption_model_processor['processor']
    device = model.device
    if not prompt:
        prompt = '<CAPTION>' if 'florence' in model.config.name_or_path else 'The image shows'

    generated = []
    for i in range(0, len(crops), batch_size):
        batch  = crops[i:i+batch_size]
        inputs = processor(images=batch, text=[prompt]*len(batch),
                           return_tensors='pt').to(
            device=device,
            dtype=torch.float16 if device.type == 'cuda' else torch.float32)
        if 'florence' in model.config.name_or_path:
            ids = model.generate(
                input_ids=inputs['input_ids'],
                pixel_values=inputs['pixel_values'],
                max_new_tokens=100, num_beams=3, do_sample=False)
        else:
            ids = model.generate(**inputs, max_length=100, num_beams=5,
                                 no_repeat_ngram_size=2, early_stopping=True)
        texts = processor.batch_decode(ids, skip_special_tokens=True)
        generated.extend([t.strip() for t in texts])
    return generated


# ── Annotate ─────────────────────────────────────────────────────────────────
def annotate(image_source, boxes, logits, phrases,
             text_scale=0.4, text_padding=5, text_thickness=2, thickness=3):
    h, w, _ = image_source.shape
    boxes_px = boxes * torch.Tensor([w, h, w, h])
    xyxy = box_convert(boxes_px, 'cxcywh', 'xyxy').numpy()
    xywh = box_convert(boxes_px, 'cxcywh', 'xywh').numpy()
    detections = sv.Detections(xyxy=xyxy)
    labels = [str(p) for p in range(boxes.shape[0])]
    ann = BoxAnnotator(text_scale=text_scale, text_padding=text_padding,
                       text_thickness=text_thickness, thickness=thickness)
    frame = ann.annotate(scene=image_source.copy(), detections=detections,
                         labels=labels, image_size=(w, h))
    coords = {str(p): v for p, v in zip(phrases, xywh)}
    return frame, coords


def detect_empty_spaces(merged_elements, cols=12, rows=20, max_empty_boxes=5, min_rect_cells=2):
    grid = [[True for _ in range(cols)] for _ in range(rows)]
    cell_w = 1.0 / cols
    cell_h = 1.0 / rows
    cell_area = cell_w * cell_h
    
    for r in range(rows):
        for c in range(cols):
            cx1, cy1 = c * cell_w, r * cell_h
            cx2, cy2 = (c + 1) * cell_w, (r + 1) * cell_h
            
            for elem in merged_elements:
                bbox = elem.get('bbox', [])
                if len(bbox) != 4:
                    continue
                ex1, ey1, ex2, ey2 = bbox
                
                ix1 = max(cx1, ex1)
                iy1 = max(cy1, ey1)
                ix2 = min(cx2, ex2)
                iy2 = min(cy2, ey2)
                
                if ix2 > ix1 and iy2 > iy1:
                    inter_area = (ix2 - ix1) * (iy2 - iy1)
                    if inter_area > 0.05 * cell_area:
                        grid[r][c] = False
                        break
                        
    empty_boxes = []
    available = [[grid[r][c] for c in range(cols)] for r in range(rows)]
    
    while True:
        start_cell = None
        for r in range(rows):
            for c in range(cols):
                if available[r][c]:
                    start_cell = (r, c)
                    break
            if start_cell:
                break
        if not start_cell:
            break
            
        r_start, c_start = start_cell
        best_area = 0
        best_rect = (r_start, c_start, r_start, c_start)
        
        for r_end in range(r_start, rows):
            for c_end in range(c_start, cols):
                valid = True
                for x in range(r_start, r_end + 1):
                    for y in range(c_start, c_end + 1):
                        if not available[x][y]:
                            valid = False
                            break
                    if not valid:
                        break
                if valid:
                    area = (r_end - r_start + 1) * (c_end - c_start + 1)
                    if area > best_area:
                        best_area = area
                        best_rect = (r_start, c_start, r_end, c_end)
                else:
                    break
                    
        r1, c1, r2, c2 = best_rect
        for x in range(r1, r2 + 1):
            for y in range(c1, c2 + 1):
                available[x][y] = False
                
        if best_area >= min_rect_cells:
            xmin = c1 * cell_w
            ymin = r1 * cell_h
            xmax = (c2 + 1) * cell_w
            ymax = (r2 + 1) * cell_h
            empty_boxes.append({
                'type': 'empty',
                'bbox': [xmin, ymin, xmax, ymax],
                'interactivity': True,
                'content': 'Empty Space'
            })
            
    empty_boxes.sort(key=lambda x: (x['bbox'][2]-x['bbox'][0]) * (x['bbox'][3]-x['bbox'][1]), reverse=True)
    return empty_boxes[:max_empty_boxes]


# ── Main SOM pipeline ────────────────────────────────────────────────────────
def get_som_labeled_img(img_path, model, BOX_TRESHOLD=0.05,
                        output_coord_in_ratio=True, ocr_bbox=None,
                        draw_bbox_config=None, caption_model_processor=None,
                        ocr_text=[], use_local_semantics=True,
                        iou_threshold=0.9, scale_img=False, imgsz=640,
                        batch_size=32, prompt=None,
                        cols=12, rows=20, max_empty_boxes=5, min_rect_cells=2):
    image_source = Image.open(img_path).convert('RGB')
    w, h = image_source.size
    if not imgsz:
        imgsz = (h, w)

    xyxy, logits, phrases = predict_yolo(
        model=model, image_path=img_path,
        box_threshold=BOX_TRESHOLD, imgsz=imgsz,
        scale_img=scale_img, iou_threshold=0.1)
    xyxy   = xyxy / torch.Tensor([w, h, w, h]).to(xyxy.device)
    img_np = np.asarray(image_source)
    h_np, w_np, _ = img_np.shape

    if ocr_bbox:
        ocr_bbox_t = (torch.tensor(ocr_bbox) /
                      torch.Tensor([w_np, h_np, w_np, h_np])).tolist()
    else:
        ocr_bbox_t = None

    ocr_elems  = [{'type': 'text', 'bbox': b, 'interactivity': False, 'content': t}
                  for b, t in zip(ocr_bbox_t or [], ocr_text)]
    icon_elems = [{'type': 'icon', 'bbox': b, 'interactivity': True, 'content': None}
                  for b in xyxy.tolist()]

    merged        = remove_overlap_new(icon_elems, iou_threshold, ocr_elems)
    empty_elems   = detect_empty_spaces(merged, cols=cols, rows=rows, max_empty_boxes=max_empty_boxes, min_rect_cells=min_rect_cells)
    merged.extend(empty_elems)
    merged_sorted = sorted(merged, key=lambda x: x['content'] is None)
    start_idx     = next((i for i, b in enumerate(merged_sorted)
                          if b['content'] is None), -1)
    boxes_t       = torch.tensor([b['bbox'] for b in merged_sorted])

    if use_local_semantics and caption_model_processor:
        captions   = get_parsed_content_icon(
            boxes_t, start_idx, img_np, caption_model_processor,
            prompt=prompt, batch_size=batch_size)
        ocr_labels = [f'Text Box ID {i}: {t}' for i, t in enumerate(ocr_text)]
        icon_start = len(ocr_labels)
        cap_iter   = iter(captions)
        for b in merged_sorted:
            if b['content'] is None:
                b['content'] = next(cap_iter, '')
        icon_labels  = [f'Icon Box ID {icon_start+i}: {c}'
                        for i, c in enumerate(captions)]
        content_list = ocr_labels + icon_labels
    else:
        content_list = [f'Text Box ID {i}: {t}' for i, t in enumerate(ocr_text)]

    boxes_cx = box_convert(boxes_t, 'xyxy', 'cxcywh')
    cfg = draw_bbox_config or {
        'text_scale': 0.4, 'text_padding': 5,
        'text_thickness': 2, 'thickness': 3}
    annotated, label_coords = annotate(
        img_np, boxes_cx, logits,
        list(range(len(boxes_cx))), **cfg)

    pil_out = Image.fromarray(annotated)
    buf = io.BytesIO()
    pil_out.save(buf, format='PNG')
    encoded = base64.b64encode(buf.getvalue()).decode('ascii')

    if output_coord_in_ratio:
        label_coords = {k: [v[0]/w_np, v[1]/h_np, v[2]/w_np, v[3]/h_np]
                        for k, v in label_coords.items()}
    return encoded, label_coords, merged_sorted

print('All utility functions defined.')

Loading EasyOCR...
PaddleOCR unavailable (No module named 'paddleocr'); falling back to EasyOCR only.
All utility functions defined.


## OmniParser Class

In [17]:
class OmniParser:
    """
    Self-contained OmniParser.

    Pickle-safe: __getstate__ strips non-picklable model objects and stores
    only the config paths; __setstate__ re-loads models from those paths.
    """

    def __init__(self, yolo_model_path, caption_model_name='florence2',
                 caption_model_path=None, device=None,
                 box_threshold=0.05, iou_threshold=0.1,
                 draw_bbox_config=None, use_local_semantics=True,
                 use_paddleocr=True, imgsz=640, batch_size=32,
                 cols=12, rows=20, max_empty_boxes=5, min_rect_cells=2):

        self.yolo_model_path     = yolo_model_path
        self.caption_model_name  = caption_model_name
        self.caption_model_path  = caption_model_path
        self.device              = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.box_threshold       = box_threshold
        self.iou_threshold       = iou_threshold
        self.use_local_semantics = use_local_semantics
        self.use_paddleocr       = use_paddleocr
        self.imgsz               = imgsz
        self.batch_size          = batch_size
        self.cols                = cols
        self.rows                = rows
        self.max_empty_boxes     = max_empty_boxes
        self.min_rect_cells      = min_rect_cells
        self.draw_bbox_config    = draw_bbox_config or {
            'text_scale': 0.8, 'text_thickness': 2,
            'text_padding': 3, 'thickness': 3,
        }
        self._load_models()

    def _load_models(self):
        print(f'Loading YOLO from:\n  {self.yolo_model_path}')
        self.yolo_model = get_yolo_model(self.yolo_model_path)
        if self.use_local_semantics and self.caption_model_path:
            print(f'Loading caption model ({self.caption_model_name}) from:\n  {self.caption_model_path}')
            self.caption_model_processor = get_caption_model_processor(
                self.caption_model_name, self.caption_model_path, self.device)
        else:
            self.caption_model_processor = None
        print('Models ready.')

    # ── pickle support ──────────────────────────────────────────────────────
    def __getstate__(self):
        """Strip non-picklable model objects; keep only config paths."""
        state = self.__dict__.copy()
        state.pop('yolo_model', None)
        state.pop('caption_model_processor', None)
        return state

    def __setstate__(self, state):
        """Re-load models from stored paths on unpickle."""
        self.__dict__.update(state)
        self._load_models()

    # ── main API ────────────────────────────────────────────────────────────
    def parse(self, image_path: str):
        """
        Parse a screenshot.

        Returns
        -------
        annotated_image   : PIL.Image
        elements          : list[dict]  — {type, bbox, interactivity, content}
        label_coordinates : dict        — {id: [cx, cy, w, h]}
        """
        ocr_result, _ = check_ocr_box(
            image_path,
            output_bb_format='xyxy',
            easyocr_args={'paragraph': False, 'text_threshold': 0.9},
            use_paddleocr=self.use_paddleocr
        )
        ocr_text, ocr_bbox = ocr_result

        image_obj = Image.open(image_path)
        overlay   = image_obj.size[0] / 3200
        cfg = {
            'text_scale'    : self.draw_bbox_config.get('text_scale', 0.8) * overlay,
            'text_thickness': max(int(self.draw_bbox_config.get('text_thickness', 2) * overlay), 1),
            'text_padding'  : max(int(self.draw_bbox_config.get('text_padding', 3)  * overlay), 1),
            'thickness'     : max(int(self.draw_bbox_config.get('thickness', 3)     * overlay), 1),
        }

        encoded, label_coords, elements = get_som_labeled_img(
            image_path, self.yolo_model,
            BOX_TRESHOLD=self.box_threshold,
            output_coord_in_ratio=True,
            ocr_bbox=ocr_bbox,
            draw_bbox_config=cfg,
            caption_model_processor=self.caption_model_processor,
            ocr_text=ocr_text,
            use_local_semantics=self.use_local_semantics,
            iou_threshold=self.iou_threshold,
            scale_img=False,
            batch_size=self.batch_size,
            imgsz=self.imgsz,
            cols=self.cols,
            rows=self.rows,
            max_empty_boxes=self.max_empty_boxes,
            min_rect_cells=self.min_rect_cells,
        )
        annotated_img = Image.open(io.BytesIO(base64.b64decode(encoded)))
        return annotated_img, elements, label_coords

print('OmniParser class defined.')

OmniParser class defined.


## Instantiate OmniParser

In [18]:
import torch

PADDLE_AVAILABLE = False  # set True if PaddleOCR installed successfully

try:
    from paddleocr import PaddleOCR
    _paddle_ocr = PaddleOCR(lang='en', use_angle_cls=False, use_gpu=False,
                             show_log=False, max_batch_size=1024,
                             use_dilation=True, det_db_score_mode='slow',
                             rec_batch_num=1024)
    PADDLE_AVAILABLE = True
    print("PaddleOCR ready.")
except Exception as e:
    print(f"PaddleOCR unavailable: {e}")

parser = OmniParser(
    yolo_model_path    = '/kaggle/input/omniparser/pytorch/default/1/OmniParser/weights/icon_detect/best.pt',
    caption_model_name = 'florence2',
    caption_model_path = '/kaggle/input/omniparser/pytorch/default/1/OmniParser/weights/icon_caption_florence',
    device             = 'cuda' if torch.cuda.is_available() else 'cpu',
    box_threshold      = 0.05,
    iou_threshold      = 0.1,
    use_local_semantics= True,
    use_paddleocr      = PADDLE_AVAILABLE,
    imgsz              = 640,
    batch_size         = 32,
)
print("OmniParser ready.")

PaddleOCR unavailable: No module named 'paddleocr'
Loading YOLO from:
  /kaggle/input/omniparser/pytorch/default/1/OmniParser/weights/icon_detect/best.pt
Loading caption model (florence2) from:
  /kaggle/input/omniparser/pytorch/default/1/OmniParser/weights/icon_caption_florence


/opt/conda/lib/python3.10/site-packages/ultralytics/nn/tasks.py:714: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, map_location="cpu")


Models ready.
OmniParser ready.


## Worker Helper Functions

In [19]:
import base64
import io
import json
import os
import tempfile
import time
from PIL import Image

# ── Decode base64 image → temp file ─────────────────────────────────────────
def b64_to_tempfile(b64_string: str) -> str:
    """Write a base64 PNG string to a temp file, return file path."""
    img_bytes = base64.b64decode(b64_string)
    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    tmp.write(img_bytes)
    tmp.close()
    return tmp.name

# ── PIL Image → base64 ───────────────────────────────────────────────────────
def pil_to_b64(pil_img: Image.Image) -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('utf-8')

# ── Get one pending task from Firebase ───────────────────────────────────────
def get_pending_task():
    """
    Query the tasks/ node for the first task with status=pending.
    Returns (task_id, task_data) or (None, None).
    """
    tasks = db.reference('tasks').get()
    if not tasks:
        return None, None
    for task_id, data in tasks.items():
        if data.get('status') == 'pending':
            return task_id, data
    return None, None

# ── Mark task status ─────────────────────────────────────────────────────────
def mark_status(task_id: str, status: str):
    db.reference(f'tasks/{task_id}/status').set(status)

# ── Upload result to Firebase ────────────────────────────────────────────────
def upload_result(task_id: str, annotated_img: Image.Image, elements: list):
    elements_serializable = []
    for i, elem in enumerate(elements):
        bbox = elem.get('bbox', [])
        if hasattr(bbox, 'tolist'):
            bbox = bbox.tolist()
        elements_serializable.append({
            'ID'      : i,
            'type'    : elem.get('type'),
            'bbox'    : bbox,
            'content' : elem.get('content'),
        })

    db.reference(f'results/{task_id}').set({
        'status'          : 'done',
        'annotated_image' : pil_to_b64(annotated_img),
        'elements'        : elements_serializable,
        'completed_at'    : time.time(),
    })
    print(f"[WORKER] Result uploaded for {task_id}")

print("Worker helpers defined.")

Worker helpers defined.


## The main worker loop - will continue running

In [ ]:
import signal

RUNNING = True

def stop_handler(sig, frame):
    global RUNNING
    print("\n[WORKER] Stopping gracefully...")
    RUNNING = False

signal.signal(signal.SIGINT, stop_handler)

print("[WORKER] Starting worker loop. Press Ctrl+C to stop.")
print("[WORKER] Waiting for tasks...\n")

while RUNNING:
    task_id, task_data = get_pending_task()

    if task_id is None:
        # No pending tasks — idle
        time.sleep(2)
        continue

    print(f"[WORKER] Picked up task: {task_id}")

    try:
        # ── Step 1: Mark as processing ───────────────────────────────────
        mark_status(task_id, 'processing')

        # ── Step 2: Decode image ─────────────────────────────────────────
        img_path = b64_to_tempfile(task_data['image'])

        # ── Step 3: Run OmniParser ───────────────────────────────────────
        t0 = time.time()
        annotated_img, elements, label_coords = parser.parse(img_path)
        elapsed = time.time() - t0
        print(f"[WORKER] Parsed in {elapsed:.1f}s | {len(elements)} elements")

        # ── Step 4: Upload result ────────────────────────────────────────
        upload_result(task_id, annotated_img, elements)

        # ── Step 5: Mark task done ───────────────────────────────────────
        mark_status(task_id, 'done')

    except Exception as e:
        print(f"[WORKER] Error processing {task_id}: {e}")
        mark_status(task_id, 'error')

    finally:
        # Clean up temp file
        try:
            os.unlink(img_path)
        except Exception:
            pass

    time.sleep(1)   # brief pause before next task

print("[WORKER] Worker stopped.")

[WORKER] Starting worker loop. Press Ctrl+C to stop.
[WORKER] Waiting for tasks...

[WORKER] Picked up task: ff48c991-0d0c-4878-bc03-90314611f1b8


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):



image 1/1 /tmp/tmpywhmb5xg.png: 640x288 13 0s, 53.4ms
Speed: 4.5ms preprocess, 53.4ms inference, 225.0ms postprocess per image at shape (1, 3, 640, 288)
[WORKER] Parsed in 3.8s | 15 elements
[WORKER] Result uploaded for ff48c991-0d0c-4878-bc03-90314611f1b8
[WORKER] Picked up task: 801e9abd-1393-419a-8e70-7d61cad4357c

image 1/1 /tmp/tmpfrjt00gf.png: 640x288 13 0s, 6.4ms
Speed: 1.9ms preprocess, 6.4ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 288)
[WORKER] Parsed in 1.7s | 15 elements
[WORKER] Result uploaded for 801e9abd-1393-419a-8e70-7d61cad4357c
[WORKER] Picked up task: 2ca575d2-f8d5-4744-9614-97d04e921962

image 1/1 /tmp/tmpnwx06zss.png: 640x288 37 0s, 6.7ms
Speed: 1.7ms preprocess, 6.7ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 288)
[WORKER] Parsed in 3.6s | 46 elements
[WORKER] Result uploaded for 2ca575d2-f8d5-4744-9614-97d04e921962
[WORKER] Picked up task: 9008a7bd-6244-43b4-9e16-e66b6865c768

image 1/1 /tmp/tmpogcdbly7.png: 640x288 37 0s, 